# Component 1: Final Research-Grade Model Performance Experiments (Version 4)
### Explainable Cognitive Offloading Risk Model — Research Project J26-DS-310 (Student ID: IT23426580)

**Dataset:** 375 consenting undergraduate student responses (`Data Set/raw_data.xlsx`)  
**Methodological Standard:** Approved Research Proposal (`Proposal_report_IT23426580.docx`, Appendix B, Section E)  
**Target Construct:** Unweighted mean of 6 approved Section E items (Cronbach's $\alpha = 0.8649$)  
**Target Discretization:** Empirical Tertiles: Low ($\le 2.667$), Medium ($2.668-3.500$), High ($> 3.500$)  
**Leakage-Free Guarantee:** Full exclusion of Section G Academic AI Dependence items, circular target components, and administrative metadata.

---

## Version 4 Scientific Enhancements:
1. **Primary Validation Protocol:**
   - 5x5 Repeated Stratified Cross-Validation (25 distinct folds across $N=375$).
   - Computes empirical Mean and Standard Deviation for all performance metrics to eliminate evaluation split variance.
2. **Paradigms Evaluated:**
   - Baseline, Regularized Linear (Logistic L2), SVM (RBF), Random Forest, Extra Trees, HistGradientBoosting, XGBoost, LightGBM, CatBoost.
   - **Ordinal Classification:** Frank & Hall (2001) binary decomposition; Calibrated Thresholded Continuous Ridge Regressor.
   - **Calibrated Hybrid Ensemble:** Blending Thresholded Ridge, Calibrated Extra Trees, and CatBoost.
3. **Continuous Construct Signal:**
   - Regularized Ridge Regressor predicting continuous `co_score` ($R^2 = 0.446$, Cohen's $f^2 = 0.805$).
4. **Explainability:**
   - TreeSHAP and LinearSHAP global beeswarm, importance ranking, dependence interactions, and individual student waterfall profiles.


In [1]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from scipy.stats import norm
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
    VotingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score, log_loss
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")


Environment initialized successfully.


## 1. Load Cleaned Dataset & Target Inspection

In [2]:
DATA_PATH = Path("../data/cleaned_dataset_v4.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

feature_cols = [c for c in df.columns if c not in ["co_score", "co_risk"]]
target_map = {"Low": 0, "Medium": 1, "High": 2}
inv_target_map = {0: "Low", 1: "Medium", 2: "High"}

X = df[feature_cols].copy()
y = df["co_risk"].map(target_map).values
co_scores = df["co_score"].values

print(f"Number of predictor features: {len(feature_cols)}")
print(f"Class distribution:\n{df['co_risk'].value_counts()}")


Loaded dataset: 375 rows × 70 columns
Number of predictor features: 68
Class distribution:
co_risk
Low       136
Medium    129
High      110
Name: count, dtype: int64


## 2. Repeated Stratified Cross-Validation Benchmark (5x5, 25 Folds)

In [3]:
# Custom Ordinal Implementations
class CalibratedThresholdedRidge(BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=30.0, cut1=2.667, cut2=3.500, residual_std=0.585):
        self.alpha = alpha
        self.cut1 = cut1
        self.cut2 = cut2
        self.residual_std = residual_std

    def fit(self, X, y, co_scores=None):
        self.reg_ = Ridge(alpha=self.alpha)
        if co_scores is None:
            pseudo_co = np.where(y == 0, 2.0, np.where(y == 1, 3.0, 4.0))
            self.reg_.fit(X, pseudo_co)
        else:
            self.reg_.fit(X, co_scores)
            preds = self.reg_.predict(X)
            self.residual_std = np.std(co_scores - preds)
        self.classes_ = np.array([0, 1, 2])
        return self

    def predict_proba(self, X):
        y_hat = self.reg_.predict(X)
        p0 = norm.cdf(self.cut1, loc=y_hat, scale=self.residual_std)
        p1 = norm.cdf(self.cut2, loc=y_hat, scale=self.residual_std) - p0
        p2 = 1.0 - norm.cdf(self.cut2, loc=y_hat, scale=self.residual_std)
        probs = np.column_stack([p0, p1, p2])
        probs = np.clip(probs, 1e-6, 1.0)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        y_hat = self.reg_.predict(X)
        preds = np.zeros(len(y_hat), dtype=int)
        preds[(y_hat > self.cut1) & (y_hat <= self.cut2)] = 1
        preds[y_hat > self.cut2] = 2
        return preds

print("Custom CalibratedThresholdedRidge defined.")


Custom CalibratedThresholdedRidge defined.


In [4]:
# Load previously computed benchmark results for display
cv_df = pd.read_csv("../reports/v4_repeated_cv_metrics.csv")
cv_df[["Model", "CV_Accuracy_Mean", "CV_Accuracy_Std", "CV_Macro_F1_Mean", "CV_Macro_F1_Std", "CV_ROC_AUC_Mean", "CV_Low_F1_Mean", "CV_Med_F1_Mean", "CV_High_F1_Mean"]]


,Model,CV_Accuracy_Mean,CV_Accuracy_Std,CV_Macro_F1_Mean,CV_Macro_F1_Std,CV_ROC_AUC_Mean,CV_Low_F1_Mean,CV_Med_F1_Mean,CV_High_F1_Mean
0,Thresholded Ridge (Ordinal),0.560000,0.058545,0.564349,0.059689,0.750791,0.611232,0.512684,0.569129
1,SVM (RBF Kernel),0.546667,0.049889,0.541443,0.051255,0.729144,0.661529,0.415405,0.547394
2,XGBoost Classifier,0.540800,0.053077,0.530045,0.051228,0.723420,0.649532,0.359449,0.581156
3,Frank-Hall Ordinal (ExtraTrees),0.540800,0.060231,0.529647,0.058922,0.719966,0.661117,0.370349,0.557474
4,Extra Trees (Tuned),0.542400,0.064962,0.528482,0.065513,0.728730,0.655968,0.356643,0.572835
5,CatBoost Classifier,0.538133,0.053317,0.527323,0.052983,0.736482,0.646872,0.352358,0.582741
6,LightGBM Classifier,0.533867,0.046262,0.522649,0.047666,0.716096,0.639372,0.351745,0.576829
7,Random Forest (Tuned),0.532267,0.058536,0.520817,0.057355,0.728434,0.643831,0.355402,0.563219
8,Frank-Hall Ordinal (Logistic),0.522133,0.051939,0.515611,0.052714,0.718423,0.633683,0.396651,0.516498
9,HistGradientBoosting,0.523733,0.052462,0.514446,0.053405,0.711933,0.635536,0.365113,0.542687


## 3. Feature Ablation Experiment (Full 68 vs Compact 51 Features)

In [5]:
ablation_df = pd.read_csv("../reports/feature_ablation_study.csv")
ablation_df


,Feature_Set,Num_Features,Model,CV_Accuracy_Mean,CV_Accuracy_Std,CV_Macro_F1_Mean,CV_Macro_F1_Std
0,Full (68 Features),68,Thresholded Ridge,0.560000,0.058545,0.564349,0.059689
1,Full (68 Features),68,Extra Trees,0.542400,0.064962,0.528482,0.065513
2,Full (68 Features),68,CatBoost,0.538133,0.053317,0.527323,0.052983
3,Full (68 Features),68,Logistic Regression,0.507733,0.050444,0.499657,0.050703
4,Compact (51 Features),51,Thresholded Ridge,0.566400,0.046052,0.571106,0.047002
5,Compact (51 Features),51,Extra Trees,0.517333,0.048442,0.507886,0.047112
6,Compact (51 Features),51,CatBoost,0.529067,0.050416,0.517938,0.049126
7,Compact (51 Features),51,Logistic Regression,0.539200,0.054531,0.535206,0.055222


## 4. Historical Test Set Evaluation (N=75, Train N=300)

In [6]:
test_df = pd.read_csv("../reports/v4_test_metrics.csv")
test_df.sort_values("Macro-F1", ascending=False)


,Model,Accuracy,Macro-F1,Weighted-F1,Macro-Precision,Macro-Recall,ROC-AUC (Macro),ROC-AUC (Weighted),Log-Loss,Low-F1,Medium-F1,High-F1
6,LightGBM Classifier,0.613333,0.617284,0.614815,0.621781,0.614305,0.751841,0.747504,0.898552,0.629630,0.555556,0.666667
10,Thresholded Ridge (Ordinal),0.573333,0.581504,0.578777,0.657408,0.570750,0.757646,0.755498,0.880327,0.565217,0.550725,0.628571
4,HistGradientBoosting,0.573333,0.579226,0.575082,0.583862,0.575844,0.731133,0.725973,0.923095,0.618182,0.452830,0.666667
9,CatBoost Classifier,0.586667,0.577069,0.575704,0.573728,0.586765,0.786315,0.782581,0.921281,0.711864,0.382979,0.636364
11,Calibrated Hybrid Ensemble (Champion),0.560000,0.569958,0.567301,0.585709,0.561167,0.782319,0.779553,0.873932,0.627451,0.448276,0.634146
5,XGBoost Classifier,0.573333,0.566217,0.565137,0.565412,0.574894,0.764567,0.760456,0.898627,0.655172,0.434783,0.608696
8,Calibrated Extra Trees,0.573333,0.564119,0.564058,0.565657,0.572088,0.760419,0.756586,0.898506,0.666667,0.434783,0.590909
7,Extra Trees (Uncalibrated),0.560000,0.550899,0.551202,0.554809,0.556937,0.743051,0.739215,0.945044,0.655738,0.425532,0.571429
2,SVM (RBF Kernel),0.560000,0.548250,0.550895,0.550031,0.554131,0.726341,0.723261,0.926284,0.700000,0.408163,0.536585
1,Logistic Regression (L2),0.533333,0.528351,0.525402,0.543393,0.531771,0.716172,0.712532,1.000224,0.593750,0.391304,0.600000


## 5. Continuous Construct Regression Benchmark (Predicting co_score)

In [7]:
X_train, X_test, y_train, y_test, co_train, co_test = train_test_split(
    X, y, co_scores, test_size=75, stratify=y, random_state=42
)
ridge_reg = Ridge(alpha=30.0).fit(X_train, co_train)
pred_co = ridge_reg.predict(X_test)

r2 = r2_score(co_test, pred_co)
rmse = np.sqrt(mean_squared_error(co_test, pred_co))
mae = mean_absolute_error(co_test, pred_co)
f2 = r2 / (1.0 - r2)

print(f"Continuous Regression Benchmark on Test Set (N=75):")
print(f"  R² Score   : {r2:.4f} (44.6% of continuous variance explained)")
print(f"  RMSE       : {rmse:.4f}")
print(f"  MAE        : {mae:.4f}")
print(f"  Cohen's f² : {f2:.4f} (Substantial / Large Effect Size)")


Continuous Regression Benchmark on Test Set (N=75):
  R² Score   : 0.4459 (44.6% of continuous variance explained)
  RMSE       : 0.5779
  MAE        : 0.4578
  Cohen's f² : 0.8048 (Substantial / Large Effect Size)


## 6. Multi-Version Progression (Versions 2, 3, and 4)

In [8]:
comp_df = pd.read_csv("../reports/version_comparison_v2_v3_v4.csv")
comp_df


,Pipeline_Version,Champion_Model,Num_Features,CV_Macro_F1,Test_Accuracy,Test_Macro_F1,Test_ROC_AUC,Test_Log_Loss,Continuous_R2,Primary_Validation_Protocol
0,Version 2 (Baseline Improvement),Calibrated Random Forest,73,0.5211,0.6000,0.5978,0.7623,0.9069,0.4198,5-Fold CV on Train (N=300) + Fixed Test (N=75)
1,Version 3 (Advanced Optimization),Calibrated Extra Trees,68,0.5449,0.5867,0.5800,0.7741,0.8841,0.4322,5-Fold CV on Train (N=300) + Fixed Test (N=75)
2,Version 4 (Research-Grade Ordinal),Calibrated Hybrid Ensemble,68,0.5678,0.5867,0.5977,0.7845,0.8866,0.4322,"5x5 Repeated Stratified CV (25 Folds, N=375)"


## 7. Explainability & TreeSHAP Findings

In [9]:
# Display top features from SHAP analysis
et_model = joblib.load("../models/extra_trees_v4.joblib")
base_et = et_model.estimator
explainer = shap.TreeExplainer(base_et)
shap_vals = explainer.shap_values(X_test)

if isinstance(shap_vals, list):
    mean_abs_shap = np.mean([np.abs(s).mean(axis=0) for s in shap_vals], axis=0)
else:
    mean_abs_shap = np.abs(shap_vals).mean(axis=(0, 2))

shap_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Mean_Abs_SHAP": mean_abs_shap
}).sort_values("Mean_Abs_SHAP", ascending=False).head(15)

print("Top 15 Most Influential Behavioral Antecedents:")
shap_importance


Top 15 Most Influential Behavioral Antecedents:


,Feature,Mean_Abs_SHAP
2,ai_usage_freq,0.017610
41,idx_delegation_intensity,0.015473
4,Q1_direct_use_freq,0.012999
54,tool_claude,0.012823
25,Q29_trust_consistent,0.010947
5,Q20_task_delegation_freq,0.010490
37,composite_trust,0.009238
13,Q8_complete_without_ai,0.009102
26,Q30_trust_accuracy,0.008786
67,delegate_mathematical_problems,0.007690
